# 🚀 Data Engineering Bootcamp: Web Scraping & API

Welcome to the interactive module! In this session, you will learn how to extract data from the internet using two primary methods: **Web Scraping** (parsing HTML directly from websites) and **API** (communicating with web services via structured endpoints).

---

## ⚙️ Preparation (Run this cell first)
We need to install the required libraries (BeautifulSoup, Selenium) and setup Google Chrome for our virtual browser.

In [ ]:
# 1. Install Python Libraries
!pip install beautifulsoup4 selenium webdriver-manager requests pandas

# 2. Install Google Chrome Stable (for Selenium Headless)
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google.list
!apt-get -y update
!apt-get install -y google-chrome-stable

print("\n[SUCCESS] Environment is ready!")

---

## 📚 Step 1: Web Scraping vs API Concepts

- **Web Scraping**: Extracting data by reading the raw HTML code of a webpage. Useful when websites do not provide direct access to their data.
- **API (Application Programming Interface)**: A formal bridge to request structured data (usually in JSON format) directly from a server without reading the visual HTML.

---

## 🏗️ Step 2 & 3: HTML Structure & DOM Tree Navigation
Before we scrape, we must understand HTML tags, attributes (`id`, `class`), and how elements relate to each other (Parent, Child, Sibling) in the **Document Object Model (DOM)**.

In [ ]:
from bs4 import BeautifulSoup

# Dummy HTML Structure
html_doc = """
<html>
  <body>
    <div id="product-list">
      <h2>Available Laptops</h2>
      <div class="item" id="item-1">
        <span class="brand">Apple</span>
        <span class="price">$1200</span>
      </div>
      <div class="item" id="item-2">
        <span class="brand">Dell</span>
        <span class="price">$800</span>
      </div>
    </div>
  </body>
</html>
"""

soup = BeautifulSoup(html_doc, 'html.parser')

print("--- HTML Attributes ---")
first_item = soup.find('div', class_='item')
print("Found Item ID:", first_item['id'])

print("\n--- DOM Tree Navigation ---")
brand_element = first_item.find('span', class_='brand')
print("Current Element:", brand_element.text)
print("Parent Element Tag:", brand_element.parent.name)
print("Next Sibling Element (Price):", brand_element.find_next_sibling('span').text)

---

## 🕸️ Step 4: Static Web Scraping with BeautifulSoup
Extracting job postings from a static HTML website (Hacker News).

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://news.ycombinator.com/jobs"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, 'html.parser')
job_rows = soup.find_all('tr', class_='athing')

data_jobs = []
for row in job_rows[:5]:
    title_element = row.find('span', class_='titleline').find('a')
    if title_element:
        data_jobs.append({
            "job_title": title_element.text,
            "url": title_element['href']
        })

df_jobs = pd.DataFrame(data_jobs)
print("--- Extracted Static Data ---")
print(df_jobs)

---

## 🤖 Step 5: Dynamic Web Scraping with Selenium
Simulating a human interacting with a modern JavaScript-rendered search engine.

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

options = webdriver.ChromeOptions()
options.add_argument('--headless=new')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--window-size=1920,1080')

print("---> Starting Headless Browser...")
try:
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    
    url = "https://hn.algolia.com/"
    print(f"---> Visiting {url}")
    driver.get(url)
    time.sleep(3)
    
    print("---> Typing 'Data Engineer' in search box...")
    search_input = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, "//input[@type='search']"))
    )
    search_input.clear()
    search_input.send_keys("Data Engineer")
    search_input.send_keys(Keys.RETURN)
    
    time.sleep(5) # Wait for JS to render results
    
    items = driver.find_elements(By.XPATH, "//article | //div[contains(@class, 'Story')]")
    print(f"\n[OK] Found {len(items)} results!")
    
    for idx, item in enumerate(items[:3]):
        text = item.text.strip().replace('\n', ' | ')
        if text:
            print(f"{idx+1}. {text[:100]}...")
            
except Exception as e:
    print("Error:", e)
finally:
    if 'driver' in locals():
        driver.quit()
        print("\n[DONE] Browser closed.")

---

## 🔌 Step 6 & 7: API Concepts, Endpoints, and HTTP Methods
APIs are accessed via **Endpoints** (URLs). You communicate your intent using **HTTP Methods**:
- `GET`: Retrieve data
- `POST`: Create new data
- `PUT` / `PATCH`: Update data
- `DELETE`: Remove data

We will use `jsonplaceholder.typicode.com`, a free fake API for testing.

In [ ]:
import requests
import json

BASE_URL = "https://jsonplaceholder.typicode.com/users"

# 1. GET Method (Retrieve Data)
print("--- 1. GET Request ---")
response_get = requests.get(f"{BASE_URL}/2")
print("Status Code:", response_get.status_code)
print("Data:", response_get.json()['name'])

# 2. POST Method (Create Data)
print("\n--- 2. POST Request ---")
payload = {"name": "John Doe", "job": "Data Engineer"}
response_post = requests.post(BASE_URL, json=payload)
print("Status Code:", response_post.status_code)
print("Response:", response_post.json())

# 3. PUT Method (Update Data)
print("\n--- 3. PUT Request ---")
payload_update = {"name": "John Doe", "job": "Senior Data Engineer"}
response_put = requests.put(f"{BASE_URL}/2", json=payload_update)
print("Status Code:", response_put.status_code)
print("Response:", response_put.json())

# 4. DELETE Method (Delete Data)
print("\n--- 4. DELETE Request ---")
response_delete = requests.delete(f"{BASE_URL}/2")
print("Status Code:", response_delete.status_code)

---

## 🚦 Step 8: Status Codes & Data Formats
- **2xx**: Success (200 OK, 201 Created)
- **4xx**: Client Error (400 Bad Request, 404 Not Found)
- **5xx**: Server Error (500 Internal Server Error)

Data returned by APIs is typically formatted in **JSON** (JavaScript Object Notation), which Python translates into native Dictionaries.

In [ ]:
# Demonstrating a 404 Not Found Error
print("--- Triggering a 404 Error ---")
response_error = requests.get("https://jsonplaceholder.typicode.com/users/999999") # User does not exist

status = response_error.status_code
print(f"API Returned Status Code: {status}")

if status == 200:
    print("Success! Data:", response_error.json())
elif status == 404:
    print("Handling Error: Data was not found on the server (404)!")
else:
    print("Unknown Error Occurred.")